In [1]:
import sys
import json
import gc
import torch

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from einops import rearrange

from torchmetrics.functional import structural_similarity_index_measure as ssim, spearman_corrcoef as spearman

sys.path.append('../data')
from data import get_train_dataloaders
from intensity import get_mints, get_spearman

from omegaconf import OmegaConf
from taming.models.vqgan import VQModel

def load_config(config_path):
    print('Enter config path')
    return OmegaConf.load(config_path)

def load_vqgan(config, ckpt_path=None):
    model = VQModel(**config.model.params)
    if ckpt_path is not None:
        sd = torch.load(ckpt_path, map_location="cpu")["state_dict"]
        model.load_state_dict(sd, strict=False)
    return model.eval()

def reconstruct(x, model):
    z, _, [_, _, indices] = model.encode(x)
    xrec = model.decode(z)
    return xrec,indices

/home/groups/ChangLab/simsz/conda/envs/vqgan/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load VQGAN

In [12]:
# Load VQGAN model
#ckpt_path = '/home/groups/ChangLab/simsz/latent-diffusion/src/taming-transformers/vqgan-aced-ckpts/epoch=000001.ckpt'
#config_path = '/home/groups/ChangLab/simsz/latent-diffusion/src/taming-transformers/configs/custom_vqgan-he.yaml'

#config_path = '/home/groups/ChangLab/govindsa/taming-transformers/configs/custom_vqgan_if_he-256.yaml'
#config_path = '/home/groups/ChangLab/govindsa/taming_transformers_backup/taming-transformers/configs/custom_vqgan.yaml'
config_path = '/home/groups/ChangLab/govindsa/checkpoints_IF/custom_vqgan_if_256.yaml'

#ckpt_path = "/home/groups/ChangLab/govindsa/taming-transformers/orion-he-if-256-vqgan-ckpts/epoch=000005.ckpt"
#ckpt_path = '/home/groups/ChangLab/govindsa/taming_transformers_backup/taming-transformers/vqgan-orion-ckpts/last.ckpt'
ckpt_path = '/home/groups/ChangLab/govindsa/checkpoints_IF/epoch=000004-256.ckpt'

config = load_config(config_path)   # use passed config_path argument
device = 'cuda:4'
model = load_vqgan(config, ckpt_path=ckpt_path).to(device)  # use passed ckpt_path argument

Enter config path
Working with z of shape (1, 256, 4, 4) = 4096 dimensions.
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.


# Get Dataloader

In [13]:
BATCH_SIZE = 200
NUM_VAL_SAMPLES = 1000
train_file = '/mnt/scratch/ORION-CRC-Unnormalized/orion_crc_dataset_sid=CRC05.h5'
val_file = '/mnt/scratch/ORION-CRC-Unnormalized/orion_crc_dataset_sid=CRC05.h5'
train_loader, val_loader = get_train_dataloaders(train_file, 
                                                 val_file, 
                                                 BATCH_SIZE, 
                                                 NUM_VAL_SAMPLES, 
                                                 remove_he=True, 
                                                 remove_background=False, 
                                                 rescale=False, 
                                                 deconvolve_he=True)


# Calculate SSIM and Spearman Correlations

In [14]:
def calculate_metrics(val_loader, model, device, batch_size, max_items=1000):
    # Pre-allocate only the needed amount of memory
    total_samples = min(len(val_loader) * batch_size, max_items)
    ssims = np.zeros(total_samples)
    mints = np.zeros((total_samples, 17))
    pmints = np.zeros((total_samples, 17))
    
    with torch.no_grad():  # Disable gradient computation
        for i, batch in tqdm(enumerate(val_loader)):
            if i * batch_size >= max_items:
                break
                
            gt, mask, meta = batch
            
            # Calculate how many samples we can process in this batch
            current_batch_size = min(batch_size, max_items - i * batch_size)
            if current_batch_size <= 0:
                break
                
            # Process only the needed samples
            gt = gt[:, :17, :, :].to(device)
            
            # Perform reconstruction
            pred, _ = reconstruct(rearrange(gt, 'b c h w -> (b c) 1 h w'), model)
            #pred, _ = reconstruct(gt, model)
            pred = pred.reshape(gt.shape)
            mints_, pmints_ = get_mints(gt, pred, mask, device)
            
            # Calculate SSIM and store results
            s = i * batch_size
            ssims[s:s+current_batch_size] = ssim(gt[:,:17], pred[:,:17]).cpu().numpy()
            
            mints[s:s+current_batch_size] = mints_.cpu().numpy()
            pmints[s:s+current_batch_size] = pmints_.cpu().numpy()
            
            # Clear memory
            del pred, gt, batch, mask, meta
            torch.cuda.empty_cache()
            gc.collect()
            
    return ssims, get_spearman(torch.tensor(mints),torch.tensor(pmints))

ssims, spearmans = calculate_metrics(
    val_loader=val_loader,
    model=model,
    device=device,
    batch_size=BATCH_SIZE,
    max_items=10_000
)

50it [02:10,  2.60s/it]


In [15]:
ssims.mean(), spearmans

(0.822075457572937,
 tensor([0.9796, 0.9907, 0.9951, 0.9950, 0.9930, 0.9555, 0.9880, 0.9937, 0.9902,
         0.9937, 0.9927, 0.9900, 0.9434, 0.9939, 0.9942, 0.9268, 0.9904],
        dtype=torch.float64))

In [11]:
spearmans.mean()

tensor(0.9888, dtype=torch.float64)